In [ ]:
# Example 1: wheelfile.WheelFile — write and read a minimal wheel archive.
import tempfile
import os
from pathlib import Path

from wheel.wheelfile import WheelFile

tmpdir = tempfile.mkdtemp()
wheel_path = os.path.join(tmpdir, 'sample_pkg-0.1.0-py3-none-any.whl')

# Create a source tree with a tiny package + dist-info.
src = Path(tmpdir) / 'src'
(src / 'sample_pkg').mkdir(parents=True)
(src / 'sample_pkg' / '__init__.py').write_text('x = 1\n')
distinfo = src / 'sample_pkg-0.1.0.dist-info'
distinfo.mkdir()
(distinfo / 'METADATA').write_text('Metadata-Version: 2.1\nName: sample_pkg\nVersion: 0.1.0\n')
(distinfo / 'WHEEL').write_text('Wheel-Version: 1.0\nGenerator: wheel-test\nRoot-Is-Purelib: true\nTag: py3-none-any\n')

with WheelFile(wheel_path, 'w') as wf:
    wf.write_files(str(src))

with WheelFile(wheel_path, 'r') as wf:
    names = wf.namelist()

print(f"Wrote {wheel_path} with {len(names)} entries")
assert any(n.endswith('__init__.py') for n in names)

In [ ]:
# Example 2: parse wheel filename into (name, version, build, tags).
# WHEEL_INFO_RE is a compiled re.Pattern in current wheel — call .match().
from wheel.wheelfile import WHEEL_INFO_RE

filename = 'numpy-1.26.0-cp311-cp311-manylinux_2_17_x86_64.whl'
m = WHEEL_INFO_RE.match(filename)
assert m is not None, f"Failed to parse {filename}"

groups = m.groupdict()
print(f"name       = {groups['name']}")
print(f"version    = {groups['ver']}")
print(f"build_tag  = {groups.get('build')}")
print(f"py_tag     = {groups['pyver']}")
print(f"abi_tag    = {groups['abi']}")
print(f"plat_tag   = {groups['plat']}")

assert groups['name'] == 'numpy'
assert groups['ver'] == '1.26.0'

In [ ]:
# Example 3: RECORD file handling — hashing + verification.
import tempfile
import os
from pathlib import Path

from wheel.wheelfile import WheelFile

tmpdir = tempfile.mkdtemp()
wheel_path = os.path.join(tmpdir, 'demo_pkg-1.0.0-py3-none-any.whl')

src = Path(tmpdir) / 'src'
(src / 'demo_pkg').mkdir(parents=True)
(src / 'demo_pkg' / '__init__.py').write_text('VERSION = "1.0.0"\n')
distinfo = src / 'demo_pkg-1.0.0.dist-info'
distinfo.mkdir()
(distinfo / 'METADATA').write_text('Metadata-Version: 2.1\nName: demo_pkg\nVersion: 1.0.0\n')
(distinfo / 'WHEEL').write_text('Wheel-Version: 1.0\nGenerator: wheel-test\nRoot-Is-Purelib: true\nTag: py3-none-any\n')

with WheelFile(wheel_path, 'w') as wf:
    wf.write_files(str(src))

# Re-open and read RECORD content (WheelFile writes RECORD on close).
with WheelFile(wheel_path, 'r') as wf:
    record_name = next(n for n in wf.namelist() if n.endswith('/RECORD'))
    record_text = wf.read(record_name).decode('utf-8')

print(f"RECORD ({record_name}):")
print(record_text)
assert 'sha256=' in record_text

In [ ]:
# Example 4: the wheel CLI — subcommand surface via `python -m wheel`.
# (wheel.cli was removed in wheel 0.46+; the CLI now lives behind `python -m wheel`.)
import subprocess
import sys

proc = subprocess.run([sys.executable, '-m', 'wheel', '--help'],
                      capture_output=True, text=True, check=True)
print(proc.stdout)
for cmd in ('unpack', 'pack', 'convert', 'tags', 'info', 'version'):
    assert cmd in proc.stdout, f"missing subcommand: {cmd}"

ver = subprocess.run([sys.executable, '-m', 'wheel', 'version'],
                     capture_output=True, text=True, check=True)
print(ver.stdout.strip())
assert 'wheel' in ver.stdout

In [ ]:
# Example 5: unpack round-trip with `python -m wheel unpack`.
import subprocess
import sys
import tempfile
from pathlib import Path

from wheel.wheelfile import WheelFile

tmp = Path(tempfile.mkdtemp())
whl = tmp / 'rt_pkg-2.0-py3-none-any.whl'
src = tmp / 'src'
(src / 'rt_pkg').mkdir(parents=True)
(src / 'rt_pkg' / '__init__.py').write_text('answer = 42\n')
di = src / 'rt_pkg-2.0.dist-info'
di.mkdir()
(di / 'METADATA').write_text('Metadata-Version: 2.1\nName: rt_pkg\nVersion: 2.0\n')
(di / 'WHEEL').write_text('Wheel-Version: 1.0\nGenerator: t\nRoot-Is-Purelib: true\nTag: py3-none-any\n')
with WheelFile(whl, 'w') as wf:
    wf.write_files(str(src))

dest = tmp / 'unpacked'
subprocess.run([sys.executable, '-m', 'wheel', 'unpack', str(whl), '-d', str(dest)], check=True)
unpacked_init = dest / 'rt_pkg-2.0' / 'rt_pkg' / '__init__.py'
assert unpacked_init.read_text() == 'answer = 42\n'
print(f"unpack round-trip OK: {unpacked_init}")